# TAO Refinement — a walkthrough

Tree-Alternating Optimization (TAO) refines an *already fitted* SGT **in place**, without changing its topology. It sweeps internal nodes bottom-up and replaces routing rules that improve its objective. This notebook shows how to run TAO after fitting and how to enable it during `fit`.

## What is TAO?

The native ShapeCART trainer grows a tree greedily top-down: a rule chosen high in the tree is never revisited. **TAO fixes routing rules after the fact** — for each internal node (bottom-up) it re-optimizes the rule against the samples that actually reach it. Read more on the general approach in the [Original TAO Paper.](https://faculty.ucmerced.edu/mcarreira-perpinan/papers/neurips18.pdf) 

In `sgtlearn`, we use TAO to repeatedly refine the shape functions learned, allowing for better training performance.

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sgtlearn import SGTClassifier, tao

X, y = fetch_openml("phoneme", version=1, return_X_y=True, as_frame=False, parser="liac-arff")
X = np.asarray(X, float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# Fit WITHOUT TAO (raw native ShapeCART).
model = SGTClassifier(max_depth=4, tao_n_runs=0, random_state=0).fit(X_tr, y_tr)
print(f"native only  train={model.score(X_tr, y_tr):.3f}")

# Refine the SAME model in place. Pass the SAME (X, y) used to fit — per-sample
# routing is not stored after fit, so TAO needs the data again.
tao.TAO_refine(model, X_tr, y_tr, n_runs=10)
print(f"after TAO    train={model.score(X_tr, y_tr):.3f}")

native only  train=0.819
after TAO    train=0.841


## Regularizing with `lambda_`

`lambda_` is a per-sample complexity rate. A non-constant rule must beat the constant "send everyone one way" rule by more than `lambda_ * n_samples` to survive, so raising it collapses weak splits into constants.

Use a depth-6 tree on the [German credit](https://www.openml.org/d/31) dataset (1000 rows, 20 features) to see how the TAO complexity penalty affects training and test accuracy.

In [2]:
from sklearn.datasets import fetch_openml

Xc, yc = fetch_openml("credit-g", version=1, return_X_y=True, as_frame=False, parser="liac-arff")
Xc = np.asarray(Xc, float)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=0, stratify=yc)

# Refit the untouched baseline for each lambda so the runs are independent.
for lam in (0.0, 3e-3, 1e-2, 3e-2, 1e-1):
    m = SGTClassifier(max_depth=6, tao_n_runs=0, random_state=0).fit(Xc_tr, yc_tr)
    tao.TAO_refine(m, Xc_tr, yc_tr, n_runs=10, lambda_=lam)
    print(f"lambda={lam:6.3f}: train={m.score(Xc_tr, yc_tr):.3f}  test={m.score(Xc_te, yc_te):.3f}")

lambda= 0.000: train=0.824  test=0.743
lambda= 0.003: train=0.824  test=0.743
lambda= 0.010: train=0.823  test=0.760


lambda= 0.030: train=0.800  test=0.767
lambda= 0.100: train=0.700  test=0.700


On this split, test accuracy rises from 74.3% at `lambda_=0` to 76.7% at `lambda_=0.03`, while training accuracy falls from 82.4% to 80.0%. At `lambda_=0.1`, both fall to 70.0%. When tuning `lambda_`, use validation scores to select it and reserve the test set for the final evaluation.

## Forests

TAO refines every base tree of a `RandomSGForestClassifier` / `RandomSGForestRegressor`, optionally in parallel.

In [3]:
from sgtlearn import RandomSGForestClassifier

forest = RandomSGForestClassifier(
    n_estimators=100, max_depth=3, tao_n_runs=0, random_state=0, n_jobs=-1
).fit(X_tr, y_tr)
print(f"forest before  test={forest.score(X_te, y_te):.3f}")

tao.TAO_refine(forest, X_tr, y_tr, n_runs=10)
print(f"forest after   test={forest.score(X_te, y_te):.3f}")

forest before  test=0.800


forest after   test=0.803


## How to enable TAO during fitting

By default, `fit` runs ShapeCART only (`tao_n_runs=0`). Set `tao_n_runs=10` to allow up to ten TAO sweeps after growing the tree.

In [4]:
refined = SGTClassifier(max_depth=4, tao_n_runs=10, random_state=0).fit(X_tr, y_tr)
print(f"tao_n_runs=10  test={refined.score(X_te, y_te):.3f}")

tao_n_runs=10  test=0.814
